In [1]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-ma4jxj_3/polara_426a3335244e45b0bcfcfe81cc5dd51b
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-ma4jxj_3/polara_426a3335244e45b0bcfcfe81cc5dd51b
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [1]:
%load_ext autoreload
%autoreload 2

In [52]:
from typing import Callable, Dict, List, Tuple, Any, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda_lag,load_yambda, split_and_reindex)

In [ ]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": None,
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ratings_Books.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [85]:
# ml_df = load_ml20m(config["ml-20m"]["interactions_path"], config=config["ml-20m"])
# yambda_df_retrieval = load_yambda(interactions_path="/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/interactions.parquet", config=config["yambda"])
yambda_df = load_yambda_lag(interactions_path=None, config=config["yambda"])
# amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [77]:
amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [86]:
# ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml-20m"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
# amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [87]:
yambda_train

,user_id,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,...,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_id,feedback,action_code
0,0,6300205,318,100,170,False,True,False,568.0,5.0,...,0.0,0.0,0.0,0.0,83.464789,47.592885,0.0,205876,1,2
1,0,8508655,318,55,170,False,False,False,1109.0,7.0,...,1.0,0.0,1.0,0.0,81.301172,46.525994,100.0,205876,1,0
3,0,12127075,939,97,280,False,True,False,1302.0,7.0,...,0.0,0.0,0.0,0.0,78.709677,56.202749,0.0,452447,1,2
5,0,7042790,1306,100,140,False,True,False,854.0,7.0,...,0.0,0.0,0.0,0.0,83.528103,66.121739,0.0,700950,1,2
6,0,8016090,1306,100,140,False,True,False,1053.0,7.0,...,1.0,0.0,1.0,0.0,82.294397,65.766423,100.0,700950,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22463550,8539,19676790,459997,100,200,False,True,False,156.0,3.0,...,1.0,0.0,1.0,0.0,88.358974,62.695924,100.0,1176554,1,2
22463551,8539,22063255,459997,100,200,False,True,False,308.0,3.0,...,2.0,0.0,2.0,0.0,91.746753,62.453086,100.0,1176554,1,2
22463552,8539,19416280,461249,100,205,False,True,False,94.0,3.0,...,0.0,0.0,0.0,0.0,92.914894,64.560065,0.0,418058,1,2
22463553,8539,20248090,461249,100,205,False,True,False,164.0,3.0,...,1.0,0.0,1.0,0.0,88.926829,64.467666,100.0,418058,1,2


In [80]:
yambda_test["timestamp"].min()

np.uint32(24220080)

In [81]:
amzn_train

,user_id,item_id,feedback,timestamp
0,444493,483281,5.0,1441260345000
1,444493,577400,5.0,1441260365000
2,444493,481957,5.0,1523093714024
3,444493,698872,1.0,1611623223325
4,444493,679968,3.0,1612044209266
...,...,...,...,...
29139224,356218,212777,5.0,1356707193000
29139225,356218,596951,3.0,1356707339000
29139226,356218,331473,4.0,1360425369000
29139227,356218,375732,4.0,1360425448000


In [82]:
datasets = {
    # "ml-20m": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    "amzn-books": {
        "train": amzn_train,
        "test": amzn_test,
        "desc": amzn_data_description,
    },
}

In [83]:
for ds in datasets.values():
    display(ds['train'].head())

,user_id,item_id,timestamp,feedback
4,0,161361,106910,1
5,0,107864,146130,1
6,0,87725,146130,1
7,0,10662,673635,1
8,0,66388,852920,1


,user_id,item_id,feedback,timestamp
0,444493,483281,5.0,1441260345000
1,444493,577400,5.0,1441260365000
2,444493,481957,5.0,1523093714024
3,444493,698872,1.0,1611623223325
4,444493,679968,3.0,1612044209266


In [84]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7451064,748414,8199478,0.909,0.091,68140,52492,163434
1,amzn-books,10931769,512200,11443969,0.955,0.045,1160538,215463,900095


## HSTU Amazon Books experiment

Запуск HSTU-пайплайна на Amazon Books

In [10]:
import torch

from src.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.models.hstu import HSTUModel


In [26]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["amzn-books"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=4,
    num_heads=4,
    linear_dim=16,
    attention_dim=16,
    input_dropout_rate=0.5,
    linear_dropout_rate=0.5,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=amzn_train,
    test=amzn_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=amzn_data_description["users"],
    item_col=amzn_data_description["items"],
    time_col=amzn_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


(1161643, 215463, 215463)

In [28]:
hstu_model = HSTUModel(
    num_items=amzn_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [29]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

item_embeddings: total=57,606,144, trainable=57,606,144
pos_embeddings: total=3,200, trainable=3,200
input_dropout: total=0, trainable=0
blocks: total=84,112, trainable=84,112

params_sum=57,693,456, trainable_sum=57,693,456


In [46]:
losses, eval_history = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    filter_seen=hstu_config.filter_seen,
)
hstu_losses

KeyboardInterrupt: 

In [18]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


HSTU evaluation: 100%|██████████| 1684/1684 [02:12<00:00, 12.68it/s]


{'hitrate': 0.02734576238147617,
 'recall': 0.012883055933276683,
 'ndcg': 0.0033107533984527526,
 'coverage': 0.01155766891272588}

## HSTU MovieLens experiment

Запуск HSTU-пайплайна на MovieLens

In [ ]:
hstu_config = HSTUExperimentConfig(
    max_seq_len=config["amzn-books"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=256,
    num_blocks=4,
    num_heads=4,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=128,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias = True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

hstu_train_loader, hstu_eval_loader, hstu_train_dataset, hstu_eval_dataset, hstu_targets = build_hstu_dataloaders(
    train=amzn_train,
    test=amzn_test,
    max_seq_len=hstu_config.max_seq_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=amzn_data_description["users"],
    item_col=amzn_data_description["items"],
    time_col=amzn_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)


len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


In [ ]:
hstu_model = HSTUModel(
    num_items=amzn_data_description["n_items"],
    embedding_dim=hstu_config.embedding_dim,
    max_seq_len=hstu_config.max_seq_len,
    num_blocks=hstu_config.num_blocks,
    num_heads=hstu_config.num_heads,
    linear_dim=hstu_config.linear_dim,
    attention_dim=hstu_config.attention_dim,
    num_negatives=hstu_config.num_negatives,
    softmax_temperature=hstu_config.softmax_temperature,
    sampling_strategy=hstu_config.sampling_strategy,
    user_embedding_norm=hstu_config.user_embedding_norm,
    l2_norm_embeddings=hstu_config.l2_norm_embeddings,
    l2_norm_eps=hstu_config.l2_norm_eps,
    item_id_offset=hstu_config.item_id_offset,
    input_dropout_rate=hstu_config.input_dropout_rate,
    linear_dropout_rate=hstu_config.linear_dropout_rate,
    attn_dropout_rate=hstu_config.attn_dropout_rate,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [ ]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")

In [ ]:
hstu_losses = train_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
)
hstu_losses


In [ ]:
hstu_metrics, hstu_candidates = evaluate_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=amzn_data_description["n_items"],
    topk=hstu_config.topk,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics
